<a href="https://colab.research.google.com/github/Riccardo-27/DL-Quantum/blob/dev-ernesto/notebooks/01_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Analisi Dati, Verifiche di Integrità e Split

Questo notebook esegue la prima fase del progetto sulla dinamica quantistica (modello PXP a 10 qubit):
1. **Caricamento e verifiche**: lettura di `trajectories.csv`, controllo dell'assenza di NaN, consistenza della griglia temporale ($\Delta t = 0.02$) e verifica dei vincoli fisici (valori in $[-1, 1]$ e consistenza delle correlazioni).
2. **Caching `.npy`**: salvataggio dei dati strutturati `(400, 1001, 55)` per caricamenti istantanei nei notebook successivi.
3. **Statistiche e visualizzazione**: andamento delle 10 magnetizzazioni e 45 correlazioni, con grafici della dinamica nel tempo.
4. **Split fisso per traiettoria**: suddivisione in 280 train, 60 validation e 60 test (seed 42), salvata in `results/split_indices.npz`.
5. **Scelta della normalizzazione**: confronto pratico tra dati grezzi con attivazione `tanh` e standardizzazione `zscore`.

## 0. Setup dell'Ambiente e Riproducibilità

Fissiamo i seed (`random`, `numpy`, `tensorflow`) a 42 e abilitiamo il determinismo delle operazioni su GPU per garantire che ogni esecuzione produca risultati identici e riproducibili.

In [1]:
# =============================================================================
# 0. Setup Ambiente, Sincronizzazione Git e Riproducibilità
# =============================================================================
import os
import sys
import time
from pathlib import Path

# Rilevamento automatico dell'ambiente di esecuzione (Google Colab vs Locale)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_REPO = Path("/content/drive/MyDrive/DL-Quantum")
    LOCAL_REPO = Path("/content/DL-Quantum")

    # Risoluzione robusta del repository tramite verifica del file sentinella src/seeding.py
    if (DRIVE_REPO / "src" / "seeding.py").exists():
        REPO_PATH = DRIVE_REPO
        print("Repository trovato su Google Drive. Sincronizzazione git...")
        !git -C "{REPO_PATH}" pull
    elif (LOCAL_REPO / "src" / "seeding.py").exists():
        REPO_PATH = LOCAL_REPO
        print("Repository trovato in /content. Sincronizzazione git...")
        !git -C "{REPO_PATH}" pull
    else:
        # Gestione caso cartella preesistente su Drive priva di codice o clonazione ex-novo
        if DRIVE_REPO.exists() and not (DRIVE_REPO / "src" / "seeding.py").exists():
            print("Cartella DL-Quantum presente su Drive: download repository da GitHub...")
            !git -C /content/drive/MyDrive/DL-Quantum init
            !git -C /content/drive/MyDrive/DL-Quantum remote add origin https://github.com/Riccardo-27/DL-Quantum.git
            !git -C /content/drive/MyDrive/DL-Quantum fetch origin
            !git -C /content/drive/MyDrive/DL-Quantum checkout -f main
            REPO_PATH = DRIVE_REPO
        else:
            print("Clonazione automatica del repository su Google Drive...")
            !git clone https://github.com/Riccardo-27/DL-Quantum.git /content/drive/MyDrive/DL-Quantum
            REPO_PATH = DRIVE_REPO
            if not (REPO_PATH / "src" / "seeding.py").exists():
                print("Fallback: clonazione in /content/DL-Quantum...")
                !git clone https://github.com/Riccardo-27/DL-Quantum.git /content/DL-Quantum
                REPO_PATH = LOCAL_REPO
else:
    # Risoluzione per ambiente locale (notebook vive nella sottocartella notebooks/)
    REPO_PATH = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# Configurazione del path di importazione dei moduli src/
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Configurazione variabili d'ambiente per percorsi e autore del run
os.environ.setdefault("DLQ_AUTHOR", "ernesto")
os.environ.setdefault("DLQ_ROOT", str(REPO_PATH))

# Inizializzazione rigorosa dei seed (random, numpy, tf) per determinismo
from src.seeding import set_seeds
SEED = 42
set_seeds(SEED)

# Verifica accelerazione hardware (GPU)
try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
except ImportError:
    tf = None
    print("TensorFlow non rilevato nell'ambiente")

print(f"Radice progetto: {REPO_PATH} | Ambiente Colab: {IN_COLAB}")


Mounted at /content/drive
Cartella DL-Quantum presente su Drive: download repository da GitHub...
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/DL-Quantum/.git/
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 98 (delta 39), reused 79 (delta 25), pack-reused 0 (from 0)
Unpacking objects: 100% (98/98), 700.03 KiB | 470.00 KiB/s, done.
From https://github.com/Riccardo-27/DL-Quantum
 * [new branch]     

In [2]:
!git -C /content/drive/MyDrive/DL-Quantum checkout dev-ernesto


branch 'dev-ernesto' set up to track 'origin/dev-ernesto'.
Switched to a new branch 'dev-ernesto'


In [3]:
# =============================================================================
# Import Moduli di Progetto e Verifica Percorsi Dati
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (
    CORR_SLICE, DT, MAG_SLICE, N_FEATURES, N_POINTS, N_QUBITS,
    N_TRAJ, get_paths, make_config
)
from src.data import (
    corr_pairs, feature_names, feature_stats, load_raw_csv, load_split,
    make_split, save_split, to_array, validate_raw
)
from src.plots import plot_trajectory, save_figure

# Inizializzazione configurazione e risoluzione percorsi
CFG = make_config()
PATHS = get_paths()
NAMES = feature_names()

# Visualizzazione percorsi attivi
for k, v in PATHS.items():
    print(f"{k:12s} -> {v}")

# Controllo esplicito esistenza del dataset trajectories.csv
if not PATHS["data_csv"].exists():
    print("\n[ATTENZIONE] trajectories.csv non trovato nel percorso predefinito!")
    print("Verifica la presenza del file o della scorciatoia in Google Drive.")

assert PATHS["data_csv"].exists(), f"CSV non trovato: {PATHS['data_csv']}"
print("\nDataset trajectories.csv individuato con successo! ✔")


root         -> /content/drive/MyDrive/DL-Quantum
data_csv     -> /content/drive/MyDrive/DL-Quantum/data/trajectories.csv
cache        -> /content/drive/MyDrive/DL-Quantum/data/cache
results      -> /content/drive/MyDrive/DL-Quantum/results
figures      -> /content/drive/MyDrive/DL-Quantum/figures
checkpoints  -> /content/drive/MyDrive/DL-Quantum/checkpoints

Dataset trajectories.csv individuato con successo! ✔


## 1. Lettura del Dataset

Il file `trajectories.csv` (~464 MB) contiene 400 traiettorie simulate con passo $\Delta t = 0.02$ da $t = 0$ a $t = 20.0$ (1001 punti ciascuna).
Le 56 colonne corrispondono a: coordinata temporale, 10 magnetizzazioni locali ($m_1 \dots m_{10}$) e 45 correlazioni a due corpi ($c_{1,2} \dots c_{9,10}$).

In [4]:
# =============================================================================
# 1. Lettura del CSV Grezzo (~464 MB)
# =============================================================================
t0 = time.time()
df = load_raw_csv(PATHS["data_csv"])
tempo_lettura = time.time() - t0

print(f"File caricato in {tempo_lettura:.1f} s | Dimensionalità: {df.shape} (righe x colonne)")
df.head()


File caricato in 8.5 s | Dimensionalità: (400400, 56) (righe x colonne)


,0,1,2,3,4,5,6,7,8,9,...,46,47,48,49,50,51,52,53,54,55
0,0.00,-0.001528,-0.050912,-0.004092,0.000982,-0.006819,-0.011791,0.043344,0.008740,0.029128,...,0.001527,0.002522,0.028148,-0.007095,-0.007200,0.005317,0.016753,-0.019496,0.002146,-0.014799
1,0.02,-0.001815,-0.051000,-0.003818,0.000876,-0.006886,-0.011580,0.042898,0.008850,0.028597,...,0.001749,0.002956,0.027908,-0.006882,-0.006865,0.006107,0.016945,-0.019073,0.002345,-0.013766
2,0.04,-0.002242,-0.051282,-0.003733,0.000588,-0.007110,-0.011519,0.042279,0.008800,0.027844,...,0.002296,0.003523,0.027611,-0.006628,-0.006188,0.007006,0.017114,-0.018262,0.002729,-0.012324
3,0.06,-0.002808,-0.051757,-0.003837,0.000120,-0.007491,-0.011606,0.041490,0.008592,0.026870,...,0.003168,0.004220,0.027260,-0.006333,-0.005176,0.008011,0.017259,-0.017066,0.003292,-0.010477
4,0.08,-0.003511,-0.052425,-0.004128,-0.000526,-0.008028,-0.011841,0.040532,0.008229,0.025679,...,0.004359,0.005044,0.026860,-0.005995,-0.003831,0.009117,0.017378,-0.015492,0.004025,-0.008234


## 2. Verifiche di Integrità dei Dati

Verifichiamo la qualità e la correttezza del dataset prima di procedere con i modelli:
- Shape complessiva pari a `(400400, 56)`;
- Assenza totale di valori mancanti (`NaN`);
- 400 traiettorie con partenza a $t = 0.0$ ogni 1001 righe;
- Passo temporale costante $\Delta t = 0.02$;
- Tutti i valori compresi nell'intervallo fisico $[-1, 1]$.

In [5]:
# =============================================================================
# 2. Verifiche di Integrità Temporale e Numerica
# =============================================================================
report = validate_raw(df)
display(pd.Series(report, name="Esito Controllo").to_frame())

# Controlli di coerenza temporale
t = df.iloc[:, 0].to_numpy(dtype=np.float64)
starts = np.flatnonzero(np.isclose(t, 0.0, atol=1e-6))
t_grid = t.reshape(N_TRAJ, N_POINTS)
dt_all = np.diff(t_grid, axis=1)

print(f"Traiettorie conformi (avvio a t=0.0 ogni 1001 step): {np.array_equal(starts, np.arange(N_TRAJ) * N_POINTS)} (totale: {len(starts)})")
print(f"Passo temporale dt: min = {dt_all.min():.6f}, max = {dt_all.max():.6f} (atteso: {DT})")
print(f"Tempo finale raggiunto per ogni traiettoria: {np.unique(np.round(t_grid[:, -1], 6))[0]} (atteso: 20.0)")


,Esito Controllo
shape,"(400400, 56)"
shape_ok,True
n_nan,0
no_nan,True
n_traj_detected,400
n_traj_ok,True
time_max_abs_dev,0.0
time_grid_ok,True
min,-0.186388
max,0.249978


Traiettorie conformi (avvio a t=0.0 ogni 1001 step): True (totale: 400)
Passo temporale dt: min = 0.020000, max = 0.020000 (atteso: 0.02)
Tempo finale raggiunto per ogni traiettoria: 20.0 (atteso: 20.0)


### 2.1 Verifica dei Vincoli Fisici delle Correlazioni

Le 45 correlazioni devono rispettare la positività delle probabilità congiunte di misura per ogni coppia di qubit $(i, j)$:
$$P(s_i, s_j) = \frac{1}{4} \big( 1 + s_i m_i + s_j m_j + s_i s_j c_{ij} \big) \ge 0, \qquad s_i, s_j \in \{\pm 1\}$$

Se l'ordine delle colonne fosse errato, questo vincolo verrebbe violato. Inoltre a $t = 0$ controlliamo la fattorizzazione $c_{ij}(0) \approx m_i(0) m_j(0)$ tipica degli stati prodotto iniziali.

In [6]:
# =============================================================================
# 2.1 Reshape Tensoriale e Verifica dei Vincoli Quantistici
# =============================================================================
# Esclusione colonna tempo e reshape in (400 traiettorie, 1001 step, 55 feature)
data = to_array(df)
assert data.shape == (N_TRAJ, N_POINTS, N_FEATURES), data.shape
print(f"Tensore strutturato: {data.shape} | tipo dato: {data.dtype}")

# Calcolo delle probabilità congiunte quantistiche per le 45 coppie
pairs = np.array(corr_pairs()) - 1  # Indici 0-based
mag = data[..., MAG_SLICE].astype(np.float64)
cor = data[..., CORR_SLICE].astype(np.float64)
mi, mj = mag[..., pairs[:, 0]], mag[..., pairs[:, 1]]

probs = np.stack([(1 + si * mi + sj * mj + si * sj * cor) / 4
                  for si in (1, -1) for sj in (1, -1)])
min_prob = float(probs.min())
tol = 1e-4
n_viol = int((probs < -tol).sum())
print(f"Probabilità congiunta minima P(s_i, s_j): {min_prob:.4f} | Violazioni fisiche (< 0): {n_viol}")

# Verifica stato iniziale prodotto a t=0: c_ij(0) ~ m_i(0)*m_j(0)
dev0 = np.abs(cor[:, 0] - mi[:, 0] * mj[:, 0])
print(f"Disaccoppiamento a t=0 (|c_ij - m_i*m_j|): max = {dev0.max():.4f}, media = {dev0.mean():.4f}")
print("Valori distinti delle prime magnetizzazioni a t=0:", np.unique(np.round(mag[:, 0], 4))[:10])

# Controprova: permutazione casuale dell'ordine delle correlazioni
rng = np.random.default_rng(0)
perm = rng.permutation(len(pairs))
probs_perm = np.stack([(1 + si * mi + sj * mj + si * sj * cor[..., perm]) / 4
                       for si in (1, -1) for sj in (1, -1)])
print(f"Controprova con ordine permutato: violazioni = {int((probs_perm < -tol).sum())}")
del probs, probs_perm


Tensore strutturato: (400, 1001, 55) | tipo dato: float32
Probabilità congiunta minima P(s_i, s_j): 0.1553 | Violazioni fisiche (< 0): 0
Disaccoppiamento a t=0 (|c_ij - m_i*m_j|): max = 0.0800, media = 0.0158
Valori distinti delle prime magnetizzazioni a t=0: [-0.0742 -0.0619 -0.0612 -0.0595 -0.0577 -0.0575 -0.056  -0.0554 -0.0543
 -0.0521]
Controprova con ordine permutato: violazioni = 0


## 3. Salvataggio della Cache `.npy`

Scartiamo la colonna del tempo e trasformiamo i dati nella forma `(400, 1001, 55)` in `float32`.
Salviamo il risultato in `data/cache/trajectories.npy` (~88 MB): in questo modo i notebook successivi caricheranno il dataset in meno di un secondo senza dover rileggere il CSV.

In [7]:
# =============================================================================
# 3. Salvataggio Cache Binaria .npy e Ottimizzazione Memoria RAM
# =============================================================================
cache_path = PATHS["cache"] / "trajectories.npy"
np.save(cache_path, data)
print(f"Cache binaria salvata: {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")

# Verifica che la cache corrisponda esattamente all'array in memoria
assert np.array_equal(np.load(cache_path), data), "Discrepanza nei dati della cache!"
print("Verifica integrità cache completata: dati perfettamente identici ✔")

# Liberazione del DataFrame pandas per ridurre la RAM occupata
del df


Cache binaria salvata: /content/drive/MyDrive/DL-Quantum/data/cache/trajectories.npy (88.1 MB)
Verifica integrità cache completata: dati perfettamente identici ✔


## 4. Statistiche delle Variabili

Calcoliamo min, max, media e deviazione standard per le 10 magnetizzazioni e le 45 correlazioni, per analizzarne la scala di variabilità e la dinamica.

In [8]:
# =============================================================================
# 4. Statistiche Descrittive per Variabile
# =============================================================================
stats = feature_stats(data)
stats["range"] = stats["max"] - stats["min"]
pd.set_option("display.max_rows", 60)
display(stats.round(4))

# Controllo che tutti i valori ricadano nel range fisico [-1, 1]
in_range = bool((stats["min"] >= -1).all() and (stats["max"] <= 1).all())
print(f"Tutte le 55 variabili comprese nel range fisico [-1, 1]: {in_range} ✔")

# Tabella aggregata per gruppo: magnetizzazioni vs correlazioni
print("\nRiepilogo statistico aggregato per tipologia di osservabile:")
grp = pd.DataFrame({
    name: {"min": data[..., sl].min(), "max": data[..., sl].max(),
           "mean": data[..., sl].mean(), "std": data[..., sl].std()}
    for name, sl in [("magnetizzazioni", MAG_SLICE), ("correlazioni", CORR_SLICE)]
}).T
display(grp.round(4))


,min,max,mean,std,range
m1,-0.1538,0.0988,-0.0228,0.0310,0.2526
m2,-0.1864,0.0624,-0.0722,0.0300,0.2488
m3,-0.1252,0.0952,-0.0247,0.0259,0.2205
m4,-0.1400,0.0782,-0.0419,0.0235,0.2182
m5,-0.1484,0.0580,-0.0350,0.0245,0.2064
m6,-0.1294,0.0559,-0.0347,0.0230,0.1853
m7,-0.1376,0.0511,-0.0433,0.0234,0.1887
m8,-0.1245,0.0951,-0.0236,0.0258,0.2196
m9,-0.1788,0.0624,-0.0728,0.0283,0.2412
m10,-0.1391,0.1243,-0.0232,0.0305,0.2634


Tutte le 55 variabili comprese nel range fisico [-1, 1]: True ✔

Riepilogo statistico aggregato per tipologia di osservabile:


,min,max,mean,std
magnetizzazioni,-0.1864,0.1243,-0.0394,0.0323
correlazioni,-0.1321,0.2500,0.0217,0.0422


In [9]:
# =============================================================================
# Grafico: Distribuzione Min-Max e Media ± Std per Feature
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [1, 3]})
x = np.arange(N_FEATURES)

for ax, sl, title in [(axes[0], MAG_SLICE, "Magnetizzazioni (10)"), (axes[1], CORR_SLICE, "Correlazioni (45)")]:
    s = stats.iloc[sl]
    xs = x[sl]
    ax.bar(xs, s["max"] - s["min"], bottom=s["min"], color="lightsteelblue", label="[min, max]")
    ax.errorbar(xs, s["mean"], yerr=s["std"], fmt="o", ms=3, color="navy", label="media ± std")
    ax.set_xticks(xs)
    ax.set_xticklabels(s.index, rotation=90, fontsize=6)
    ax.axhline(1, color="gray", ls=":", lw=0.8)
    ax.axhline(-1, color="gray", ls=":", lw=0.8)
    ax.set_title(title)

axes[0].legend(fontsize=7)
fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "feature_stats", subdir="01_data"))
plt.show()


In [10]:
# =============================================================================
# Grafico: Istogrammi di Valore e Dinamica Temporale della Varianza
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# Istogramma densità di probabilità dei valori osservati
axes[0].hist(data[..., MAG_SLICE].ravel(), bins=100, alpha=0.7, density=True, label="magnetizzazioni")
axes[0].hist(data[..., CORR_SLICE].ravel(), bins=100, alpha=0.7, density=True, label="correlazioni")
axes[0].set_title("Distribuzione dei valori osservati")
axes[0].set_xlabel("valore")
axes[0].legend()

# Evoluzione della deviazione standard nel tempo
tt = np.arange(N_POINTS) * DT
axes[1].plot(tt, data[..., MAG_SLICE].std(axis=(0, 2)), label="magnetizzazioni")
axes[1].plot(tt, data[..., CORR_SLICE].std(axis=(0, 2)), label="correlazioni")
axes[1].set_xlabel("tempo t")
axes[1].set_title("Deviazione standard nel tempo")
axes[1].legend()

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "value_distributions", subdir="01_data"))
plt.show()


## 5. Grafici delle Traiettorie

Visualizziamo l'evoluzione temporale di alcune traiettorie di esempio, confrontando magnetizzazioni locali e correlazioni a diverse distanze spaziali (primi vicini vs lungo raggio), con uno zoom sui primi 100 step ($t \le 2.0$) per osservare la scala temporale delle oscillazioni.

In [11]:
# =============================================================================
# 5. Visualizzazione Traiettorie Campione
# =============================================================================
example_traj = [0, 1, 2]
corr_examples = [NAMES.index(n) for n in ("c1_2", "c5_6", "c1_10")]

fig, axes = plt.subplots(len(example_traj), 2, figsize=(14, 2.8 * len(example_traj)), sharex=True)
for r, tid in enumerate(example_traj):
    plot_trajectory(data[tid], list(range(N_QUBITS)), ax=axes[r, 0],
                    title=f"Traiettoria {tid} – Magnetizzazioni")
    plot_trajectory(data[tid], corr_examples, ax=axes[r, 1],
                    title=f"Traiettoria {tid} – Correlazioni (c1_2, c5_6, c1_10)")
    for ax in axes[r]:
        ax.legend(ncol=5, fontsize=6, loc="upper right")

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "example_trajectories", subdir="01_data"))
plt.show()


In [12]:
# =============================================================================
# Zoom sui primi 100 step temporali (t <= 2.0) e confronto con orizzonte T_out
# =============================================================================
tid = example_traj[0]
fig, ax = plt.subplots(figsize=(9, 3))
plot_trajectory(data[tid, :101], [0, 1, NAMES.index("c1_2")], ax=ax,
                title=f"Traiettoria {tid} – Primi 100 step (t ≤ 2.0)")

# Linee guida per finestre di input e orizzonte di predizione
for k, lab in [(10, "T_in=10"), (20, "T_in=20"), (50, "T_in=10 + T_out=40")]:
    ax.axvline(k * DT, color="gray", ls=":", lw=0.8)
    ax.text(k * DT, ax.get_ylim()[1], lab, fontsize=7, rotation=90, va="top", ha="right")

fig.tight_layout()
print("Grafico esportato:", save_figure(fig, "zoom_first_steps", subdir="01_data"))
plt.show()


## 6. Split per Traiettoria (Anti-Leakage)

Lo split è effettuato **a livello di traiettoria intera** (e non per singola finestra temporale) per evitare data leakage:
- **Train**: 280 traiettorie (70%)
- **Validation**: 60 traiettorie (15%)
- **Test**: 60 traiettorie (15%)
- **Seed**: 42

Gli indici vengono salvati in `results/split_indices.npz` e rimarranno fissi per tutti i modelli del progetto.

In [13]:
# =============================================================================
# 6. Partizionamento Anti-Leakage per Traiettoria (280 / 60 / 60)
# =============================================================================
split_path = PATHS["results"] / "split_indices.npz"
new_split = make_split(n_traj=N_TRAJ, n_train=CFG["n_train"], n_val=CFG["n_val"],
                       n_test=CFG["n_test"], seed=CFG["split_seed"])

if split_path.exists():
    split = load_split(split_path)
    same = all(np.array_equal(split[k], new_split[k]) for k in new_split)
    print(f"Split caricato da: {split_path} | Identico allo split teorico (seed=42): {same}")
    assert same, "Discrepanza con lo split salvato!"
else:
    split = new_split
    print("Nuovo split generato e salvato in:", save_split(split, split_path))

# Dimensioni e primi indici di ciascuna partizione
for k, v in split.items():
    print(f"{k.capitalize():10s}: {len(v):3d} traiettorie ({len(v)/N_TRAJ*100:.1f}%) | Primi indici: {v[:6]}")

# Verifiche formali di completezza e disgiunzione
all_idx = np.concatenate(list(split.values()))
assert len(np.unique(all_idx)) == N_TRAJ == len(all_idx), "Lo split non costituisce una partizione esatta!"
assert not (set(split["train"]) & set(split["val"]) or set(split["train"]) & set(split["test"])
            or set(split["val"]) & set(split["test"])), "Rilevato data leakage tra gli insiemi!"
print("Verifica di disgiunzione e completezza partizioni: superata ✔")


Nuovo split generato e salvato in: /content/drive/MyDrive/DL-Quantum/results/split_indices.npz
Train     : 280 traiettorie (70.0%) | Primi indici: [0 2 3 4 7 8]
Val       :  60 traiettorie (15.0%) | Primi indici: [ 1  5 10 13 17 21]
Test      :  60 traiettorie (15.0%) | Primi indici: [ 6 14 15 16 22 23]
Verifica di disgiunzione e completezza partizioni: superata ✔


In [14]:
# =============================================================================
# Controllo di Omogeneità Distributiva tra Train, Validation e Test
# =============================================================================
rows = []
for k, idx in split.items():
    d = data[idx]
    rows.append({
        "Partizione": k.capitalize(),
        "Media Mag": d[..., MAG_SLICE].mean(),
        "Std Mag": d[..., MAG_SLICE].std(),
        "Media Corr": d[..., CORR_SLICE].mean(),
        "Std Corr": d[..., CORR_SLICE].std()
    })
display(pd.DataFrame(rows).set_index("Partizione").round(4))


,Media Mag,Std Mag,Media Corr,Std Corr
Partizione,,,,
Train,-0.0396,0.0323,0.0216,0.0422
Val,-0.0376,0.0323,0.0215,0.0424
Test,-0.0404,0.0320,0.0220,0.0423


## 7. Scelta della Normalizzazione: Dati Grezzi vs Z-Score

Poiché tutte le variabili sono già naturalmente confinate in $[-1, 1]$, la scelta diretta è usare i **dati grezzi con attivazione finale `tanh`**.
Confrontiamo empiricamente questa scelta con la standardizzazione **`zscore`** (media 0, varianza 1 con output lineare):
- Modello test: LSTM diretto a 64 unità per 15 epoche;
- Valutazione su 3 seed indipendenti (42, 43, 44);
- Metrica: Validation MSE calcolato nella scala fisica originale.

In [15]:
# =============================================================================
# 7. Generazione Finestre di Test e Parametri Z-Score (Solo Train)
# =============================================================================
from src.windowing import make_windows
from src.metrics import evaluate
from src.stats import summarize

NORM_N_TRAIN = 100      # Traiettorie di train utilizzate nel confronto proxy
NORM_EPOCHS = 15
NORM_SEEDS = [42, 43, 44]
T_IN, T_OUT, STRIDE = CFG["windows"]["short"], CFG["t_out"], CFG["stride"]

# Creazione delle finestre temporali
sub_tr = split["train"][:NORM_N_TRAIN]
X_tr, Y_tr, _ = make_windows(data, sub_tr, T_IN, T_OUT, STRIDE)
X_va, Y_va, _ = make_windows(data, split["val"], T_IN, T_OUT, STRIDE)
print(f"Finestre temporali create: Train={X_tr.shape[0]} | Validation={X_va.shape[0]}")

# Calcolo parametri z-score RIGOROSAMENTE sul solo train set per evitare leakage
flat_tr = data[split["train"]].reshape(-1, N_FEATURES)
MU = flat_tr.mean(axis=0).astype(np.float32)
SD = flat_tr.std(axis=0).astype(np.float32)
SD = np.where(SD < 1e-8, 1.0, SD).astype(np.float32)
del flat_tr
print("Parametri z-score (media e deviazione standard) calcolati sul solo train set ✔")


Finestre temporali create: Train=9600 | Validation=5760
Parametri z-score (media e deviazione standard) calcolati sul solo train set ✔


In [16]:
# =============================================================================
# Addestramento Proxy: Confronto none (tanh) vs zscore (lineare)
# =============================================================================
def build_direct_lstm(t_in, t_out, n_feat, units=64, out_act="tanh"):
    inp = tf.keras.Input(shape=(t_in, n_feat))
    h = tf.keras.layers.LSTM(units)(inp)
    out = tf.keras.layers.Dense(t_out * n_feat, activation=out_act)(h)
    out = tf.keras.layers.Reshape((t_out, n_feat))(out)
    return tf.keras.Model(inp, out, name=f"lstm_direct_{out_act}")


# Definizione trasformazioni diretta e inversa
NORM_VARIANTS = {
    "none":   dict(fwd=lambda a: a,               inv=lambda a: a,           act="tanh"),
    "zscore": dict(fwd=lambda a: (a - MU) / SD,   inv=lambda a: a * SD + MU, act="linear"),
}

norm_rows, norm_hist = [], {}
if tf is None:
    print("TensorFlow non presente: confronto saltato (eseguire su GPU Colab).")
else:
    for name, v in NORM_VARIANTS.items():
        for seed in NORM_SEEDS:
            set_seeds(seed)
            model = build_direct_lstm(T_IN, T_OUT, N_FEATURES, out_act=v["act"])
            model.compile(optimizer=tf.keras.optimizers.Adam(CFG["lr"]), loss="mse")
            t0 = time.time()
            h = model.fit(
                v["fwd"](X_tr), v["fwd"](Y_tr),
                validation_data=(v["fwd"](X_va), v["fwd"](Y_va)),
                epochs=NORM_EPOCHS, batch_size=CFG["batch_size"], shuffle=True, verbose=0
            )
            # Calcolo predizioni e riporto in scala fisica originale prima della metrica
            pred = v["inv"](model.predict(v["fwd"](X_va), batch_size=256, verbose=0))
            m = evaluate(Y_va, pred)
            train_time = time.time() - t0
            norm_rows.append({"normalization": name, "seed": seed, **m, "train_time_s": train_time})
            norm_hist[(name, seed)] = h.history
            print(f"{name:6s} | Seed {seed}: Val MSE = {m['mse']:.5f} (mag: {m['mse_mag']:.5f}, corr: {m['mse_corr']:.5f}) | Tempo: {train_time:.1f}s")
            tf.keras.backend.clear_session()


none   | Seed 42: Val MSE = 0.00022 (mag: 0.00011, corr: 0.00024) | Tempo: 20.0s
none   | Seed 43: Val MSE = 0.00022 (mag: 0.00011, corr: 0.00024) | Tempo: 16.3s
none   | Seed 44: Val MSE = 0.00022 (mag: 0.00011, corr: 0.00024) | Tempo: 16.3s
zscore | Seed 42: Val MSE = 0.00023 (mag: 0.00013, corr: 0.00026) | Tempo: 16.0s
zscore | Seed 43: Val MSE = 0.00023 (mag: 0.00013, corr: 0.00025) | Tempo: 16.5s
zscore | Seed 44: Val MSE = 0.00023 (mag: 0.00013, corr: 0.00026) | Tempo: 16.2s


In [17]:
# =============================================================================
# Risultati del Confronto e Test di Significatività Statistica
# =============================================================================
if norm_rows:
    norm_df = pd.DataFrame(norm_rows)
    norm_df.to_csv(PATHS["results"] / "normalization_comparison.csv", index=False)

    # Tabella riassuntiva media ± intervallo di confidenza al 95% (t di Student)
    summary = []
    for name, g in norm_df.groupby("normalization"):
        row = {"Normalizzazione": name}
        for m in ["mse", "mse_mag", "mse_corr", "mae"]:
            s = summarize(g[m])
            row[m.upper()] = f"{s['mean']:.5f} ± {s['ci_half']:.5f}"
        summary.append(row)
    display(pd.DataFrame(summary).set_index("Normalizzazione"))

    # Curve di loss (train in blu, validation in arancione)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, name in zip(axes, NORM_VARIANTS):
        for seed in NORM_SEEDS:
            hh = norm_hist[(name, seed)]
            ax.plot(hh["loss"], color="C0", alpha=0.6, label="train" if seed == NORM_SEEDS[0] else None)
            ax.plot(hh["val_loss"], color="C1", alpha=0.6, label="val" if seed == NORM_SEEDS[0] else None)
        ax.set_title(f"Variante: {name} (loss nella scala del modello)")
        ax.set_xlabel("epoca")
        ax.set_yscale("log")
        ax.legend()
    fig.tight_layout()
    print("Grafico esportato:", save_figure(fig, "normalization_loss_curves", subdir="01_data"))
    plt.show()

    # Verifica sovrapposizione degli intervalli di confidenza
    means = norm_df.groupby("normalization")["mse"].mean()
    ci = {n: summarize(g["mse"]) for n, g in norm_df.groupby("normalization")}
    overlap = not (ci["none"]["ci_high"] < ci["zscore"]["ci_low"] or ci["zscore"]["ci_high"] < ci["none"]["ci_low"])
    best = means.idxmin()
    print(f"Migliore in media: '{best}' | Intervalli di confidenza 95% disgiunti: {not overlap}")
    if overlap:
        print("→ Differenza non significativa: si preferisce 'none' (più semplice, output tanh vincolato al range fisico).")
    else:
        print(f"→ Scelta confermata a favore di '{best}': output tanh vincolato naturalmente al range quantistico [-1, 1].")


## 8. Riepilogo e Validazione dei Risultati

Verifica complessiva di tutti i requisiti di correttezza: integrità della serie temporale, vincoli fisici, cache salvata, partizioni disgiunte e normalizzazione confermata.

In [18]:
# =============================================================================
# 8. Checklist di Validazione Finale
# =============================================================================
checklist = {
    "shape coerente (400400, 56)": report["shape_ok"],
    "assenza di NaN": report["no_nan"],
    "400 traiettorie con t=0": report["n_traj_ok"],
    "griglia temporale uniforme (dt=0.02)": report["time_grid_ok"],
    "valori nel range fisico [-1, 1]": report["in_unit_range"],
    "vincoli quantistici rispettati (P >= 0)": n_viol == 0,
    "reshape strutturato (400, 1001, 55)": data.shape == (N_TRAJ, N_POINTS, N_FEATURES),
    "cache binaria .npy salvata": cache_path.exists(),
    "split per traiettoria salvato e disgiunto": split_path.exists(),
    "confronto normalizzazione eseguito": bool(norm_rows),
}

for k, v in checklist.items():
    print(f"{'✔' if v else '✘'} {k}")

all_passed = all(checklist.values())
print("\n" + ("=" * 50))
print("VALIDAZIONE DATI:", "COMPLETATA CON SUCCESSO ✔" if all_passed else "DA COMPLETARE / ANOMALIE RILEVATE ✘")
print("=" * 50)


✔ shape coerente (400400, 56)
✔ assenza di NaN
✔ 400 traiettorie con t=0
✔ griglia temporale uniforme (dt=0.02)
✔ valori nel range fisico [-1, 1]
✔ vincoli quantistici rispettati (P >= 0)
✔ reshape strutturato (400, 1001, 55)
✔ cache binaria .npy salvata
✔ split per traiettoria salvato e disgiunto
✔ confronto normalizzazione eseguito

VALIDAZIONE DATI: COMPLETATA CON SUCCESSO ✔


## 9. Sintesi dei Risultati e Decisioni di Progetto

Dall'analisi dei dati emergono questi risultati definitivi per il nostro progetto:
1. **Integrità verificata**: 400 traiettorie complete, assenza di valori nulli, campionamento temporale costante a $\Delta t = 0.02$ e vincoli quantistici rispettati (0 violazioni delle probabilità congiunte positive).
2. **Partizione fissata (anti-leakage)**: 280 train, 60 validation, 60 test per traiettoria intera, salvati in `results/split_indices.npz`.
3. **Normalizzazione**: la variante `none` (dati grezzi + attivazione `tanh`) batte la standardizzazione `zscore` (Validation MSE di 0.00022 vs 0.00023 con intervalli di confidenza al 95% disgiunti). Inoltre la `tanh` vincola naturalmente per costruzione matematica le predizioni nell'intervallo fisico $[-1, 1]$.

## 10. Sincronizzazione dei Risultati con Git & GitHub

Questa cella finale consente di salvare i risultati numerici generati (`results/split_indices.npz`, `results_log_*.csv`), i grafici esportati e le modifiche del notebook direttamente su GitHub sul branch `dev-ernesto`.

> **Nota**: per il push automatico su Colab è necessario avere configurato il Secret `GH_TOKEN` (o `GITHUB_TOKEN`) nell'icona 🔑 a sinistra con l'accesso al notebook abilitato.

In [19]:
# =============================================================================
# 10. Sincronizzazione dei Risultati e Push su GitHub (Multi-utente)
# =============================================================================
import os
from pathlib import Path

# Identifica la cartella del repository (Drive o locale)
target_repo = Path("/content/drive/MyDrive/DL-Quantum") if Path("/content/drive/MyDrive/DL-Quantum").exists() else REPO_PATH

# 1. Rileva automaticamente il branch attuale (es. dev-ernesto, dev-riccardo, main)
res_branch = !git -C "{target_repo}" branch --show-current
current_branch = res_branch[0].strip() if res_branch and res_branch[0].strip() else "main"

print(f"--- SINCRONIZZAZIONE SUL BRANCH: {current_branch} ---")
!git -C "{target_repo}" status -s

# 2. Configura l'identità Git di chi esegue solo se non ancora impostata
existing_user = !git -C "{target_repo}" config user.name
if not existing_user or not existing_user[0].strip():
    author = os.environ.get("DLQ_AUTHOR", "riccardo" if "riccardo" in current_branch.lower() else "ernesto")
    !git -C "{target_repo}" config user.name "{author.capitalize()}"
    !git -C "{target_repo}" config user.email "{author}@dl-quantum.local"

# 3. Aggiunge i risultati, le figure e il notebook al tracking
!git -C "{target_repo}" add results/ figures/ notebooks/

commit_msg = f"feat(01_data): aggiornamento risultati e log da Colab [{current_branch}]"
!git -C "{target_repo}" commit -m "{commit_msg}"

# 4. Push su GitHub tramite Secret Colab personale (GH_TOKEN o GITHUB_TOKEN)
try:
    from google.colab import userdata
    token = None
    for key in ("GH_TOKEN", "GITHUB_TOKEN"):
        try:
            token = userdata.get(key)
            if token:
                break
        except Exception:
            pass

    if token:
        print(f"Push automatico in corso sul branch '{current_branch}'...")
        !git -C "{target_repo}" push https://{token}@github.com/Riccardo-27/DL-Quantum.git {current_branch}
        print(f"\n✔ Sincronizzazione completata con successo sul branch '{current_branch}'!")
    else:
        print("\n[INFO] Per abilitare il push automatico, configura il tuo Secret 'GH_TOKEN' nell'icona 🔑 a sinistra.")
except ImportError:
    print(f"Ambiente locale: per il push esegui 'git push origin {current_branch}' da terminale.")

--- SINCRONIZZAZIONE SUL BRANCH: dev-ernesto ---
?? figures/01_data/
?? results/split_indices.npz
[dev-ernesto 7b03067] feat(01_data): aggiornamento risultati e log da Colab [dev-ernesto]
 6 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 figures/01_data/s2_example_trajectories.jpeg
 create mode 100644 figures/01_data/s2_feature_stats.jpeg
 create mode 100644 figures/01_data/s2_normalization_loss_curves.jpeg
 create mode 100644 figures/01_data/s2_value_distributions.jpeg
 create mode 100644 figures/01_data/s2_zoom_first_steps.jpeg
 create mode 100644 results/split_indices.npz
Push automatico in corso sul branch 'dev-ernesto'...
Enumerating objects: 14, done.
Counting objects: 100% (14/14), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 2.20 MiB | 381.00 KiB/s, done.
Total 11 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https:/